In [ ]:
# UltimateDocResearcher — AI engineering

In [ ]:
import subprocess, os, sys
print("🚀 Starting UltimateDocResearcher research loop...")

# 1. Clone & Setup
repo = "Amitro123/UltimateDocResearcher"
print(f"Cloning repo: {repo}")

if not os.path.exists("ultimate-doc-researcher"):
    try:
        subprocess.run(["git", "clone", f"https://github.com/{repo}.git", "ultimate-doc-researcher"], check=True)
    except Exception as e:
        print(f"❌ Clone failed: {e}")
        # Try fallback if possible or exit
        raise

os.chdir("ultimate-doc-researcher")
sys.path.insert(0, ".")
print("✅ Repo cloned and directoy changed")


In [ ]:
# 2. Install Dependencies
print("📦 Installing dependencies (this may take 2-3 minutes)...")
subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "unsloth", "trl", "peft", "pymupdf",
    "aiohttp", "beautifulsoup4", "google-api-python-client", "google-auth",
    "bitsandbytes", "accelerate", "xformers",
], check=True)
print("✅ Dependencies installed")


In [ ]:
# 3. Collect & Prepare
from collector.ultimate_collector import UltimateCollector
from autoresearch.prepare import prepare
from collector.analyzer import analyze_corpus

TOPIC = "AI engineering"
N_ITERATIONS = 20

print(f"🔍 Collecting documents for: {TOPIC}")
collector = UltimateCollector(
    google_queries=[f"{TOPIC} research paper", f"{TOPIC} tutorial"],
    reddit_subreddits=["MachineLearning", "LocalLLaMA"],
    output_dir="data/",
)
docs = collector.run()
print(f"✅ Collected {len(docs)} documents")

print("🧹 Analyzing and cleaning corpus...")
report = analyze_corpus("data/all_docs.txt", "data/")
print(report)

print("📝 Preparing Q&A pairs...")
prepare(
    corpus_path="data/all_docs_cleaned.txt",
    output_dir="data/",
    program_md="templates/program.md",
    max_pairs=500,
    use_llm=False,
)
print("✅ Data preparation complete")


In [ ]:
# 4. Training Loop
from autoresearch.train import TrainConfig, train

cfg = TrainConfig(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    num_train_epochs=1,
    topic=TOPIC,
    output_dir="/kaggle/working/models/lora",
    results_tsv="/kaggle/working/results.tsv",
)

print(f"🏋️ Starting research loop ({N_ITERATIONS} iterations)...")
for i in range(N_ITERATIONS):
    cfg.iteration = i + 1
    metrics = train(cfg)
    vs = metrics.get('val_score', 'N/A')
    print(f"Iteration {i+1}/{N_ITERATIONS}: val_score={vs}")

print("✅ Training sequence complete")


In [ ]:
# 5. Export Results
import pandas as pd
if os.path.exists("/kaggle/working/results.tsv"):
    df = pd.read_csv("/kaggle/working/results.tsv", sep="\t")
    print(df[["iteration", "val_loss", "val_score", "elapsed_seconds"]])
    print("✅ Results exported")
else:
    print("⚠️ results.tsv not found")
